# IncludeHer UK - Interactive Data Explorer

Explore gender representation, mention types, subjects, and regional demographics in UK science exam specifications.

**How to use**
- Use the dropdown menus to choose a key stage, exam board, and chart view.
- **Hover** over chart segments for counts and percentages.
- **Click** a segment (or pick a demographic below) to list every named scientist in that group.

**Run with Voila** (from the `Python` folder):
```bash
pip install -r requirements-voila.txt
voila Interactive_IncludeHer_UK.ipynb
```


In [ ]:
import contextlib
import io
import os
import sys
from collections import Counter

import ipywidgets as widgets
from IPython.display import HTML, display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, os.path.abspath("."))
import Plot_Figures_UK as pf

COLOURS = pf.COLOURS
MALE_COLOUR = COLOURS[1]
FEMALE_COLOUR = COLOURS[0]
CONCEPT_COLOUR = COLOURS[3]
SCIENTIST_COLOUR = COLOURS[2]
REGION_COLOURS = pf.REGION_DONUT_COLOURS

KEY_STAGE_OPTIONS = {
    "KS5 - A-Level / Scottish Highers (ages 16-18)": "ks5",
    "KS4 - GCSE / NQ5 (ages 14-16)": "ks4",
}

VIEW_OPTIONS = [
    "Subject breakdown (by gender)",
    "Unique scientists by gender",
    "Concept vs scientist mentions",
    "Scientists by region",
    "Scientists by nationality",
    "KS4 vs KS5 region comparison",
]

DEMOGRAPHIC_OPTIONS = {
    "Gender": "gender",
    "Region": "region",
    "Nationality": "nationality",
}

SUBJECT_LABELS = {
    "environmental science": "Environmental science",
    "biology": "Biology",
    "physics": "Physics",
    "chemistry": "Chemistry",
    "geology": "Geology",
    "astronomy": "Astronomy",
}

with contextlib.redirect_stdout(io.StringIO()):
    DATA = {
        "ks5": pf.prepare_key_stage("ks5"),
        "ks4": pf.prepare_key_stage("ks4"),
    }


In [ ]:
def prettify(text):
    if not text or str(text).lower() in ("nan", "none", "unknown", ""):
        return "Unknown"
    return str(text).strip().title().replace("Eruope", "Europe")


def prettify_name(name):
    if not name:
        return ""
    parts = []
    for chunk in str(name).split(","):
        parts.append(" ".join(word.capitalize() for word in chunk.strip().split()))
    return ", ".join(parts)


def stage_cfg(stage_key):
    return DATA[stage_key]["cfg"]


def stage_dataset(stage_key):
    return DATA[stage_key]["dataset"]


def board_file_key(stage_key, board_display):
    return stage_cfg(stage_key)["boards_map"][board_display]


def board_labels(stage_key):
    return stage_cfg(stage_key)["board_labels"]


def display_board_name(stage_key, board):
    return board_labels(stage_key).get(board, board)


def subjects_for_board(stage_key, board_display):
    dataset = stage_dataset(stage_key)
    board_key = board_file_key(stage_key, board_display)
    core = {"physics", "chemistry", "biology"}
    return [
        sb
        for sb, info in dataset[board_key]["subjects"].items()
        if sb in core
        or (
            "concept" in info
            and info["concept"]["male"] + info["concept"]["female"] > 0
        )
    ]


def merge_scientists(stage_key, board_display=None):
    dataset = stage_dataset(stage_key)
    boards = stage_cfg(stage_key)["boards_display"]
    if board_display is not None:
        boards = [board_display]
    merged = {}
    for board in boards:
        board_key = board_file_key(stage_key, board)
        for name, info in dataset[board_key]["names"].items():
            if name not in merged:
                merged[name] = {
                    "gender": info.get("gender", "unknown"),
                    "region": info.get("region", "unknown"),
                    "nationality": info.get("nationality", "unknown"),
                    "mentions": info.get("number of mentions", 0),
                    "boards": [display_board_name(stage_key, board)],
                }
            else:
                label = display_board_name(stage_key, board)
                if label not in merged[name]["boards"]:
                    merged[name]["boards"].append(label)
                merged[name]["mentions"] = max(
                    merged[name]["mentions"], info.get("number of mentions", 0)
                )
    return merged


def filter_scientists(scientists, gender=None, region=None, nationality=None):
    rows = []
    for name, info in scientists.items():
        g = str(info.get("gender", "unknown")).lower()
        r = str(info.get("region", "unknown")).lower()
        n = str(info.get("nationality", "unknown")).lower()
        if gender and g != gender.lower():
            continue
        if region and r != region.lower():
            continue
        if nationality and n != nationality.lower():
            continue
        rows.append((name, info))
    return rows


def demographic_values(scientists, dimension):
    counter = Counter(str(info.get(dimension, "unknown")).lower() for info in scientists.values())
    return sorted(counter.keys(), key=lambda k: (-counter[k], k))


def demographic_counts(scientists, dimension):
    counter = Counter(str(info.get(dimension, "unknown")).lower() for info in scientists.values())
    labels = sorted(counter.keys(), key=lambda k: (-counter[k], k))
    counts = [counter[k] for k in labels]
    total = sum(counts) or 1
    display_labels = [prettify(k) for k in labels]
    hover = [
        f"{prettify(k)}<br>Count: {counter[k]}<br>Share: {counter[k] / total * 100:.1f}%"
        for k in labels
    ]
    return labels, counts, display_labels, hover, total


def names_html(rows, title):
    if not rows:
        return f"<p><em>No scientists match <strong>{title}</strong>.</em></p>"
    lines = [
        "<div style='max-height:320px; overflow-y:auto; border:1px solid #ddd; border-radius:8px; padding:12px; background:#fafafa;'>",
        f"<p style='margin-top:0'><strong>{title}</strong> - {len(rows)} scientist{'s' if len(rows) != 1 else ''}</p>",
        "<table style='width:100%; border-collapse:collapse; font-size:14px;'>",
        "<tr style='background:#eef3f8'><th align='left'>Name</th><th align='left'>Gender</th><th align='left'>Region</th><th align='left'>Nationality</th><th align='right'>Mentions</th></tr>",
    ]
    for name, info in sorted(rows, key=lambda x: prettify_name(x[0])):
        lines.append(
            "<tr style='border-top:1px solid #eee'>"
            f"<td>{prettify_name(name)}</td>"
            f"<td>{prettify(info.get('gender'))}</td>"
            f"<td>{prettify(info.get('region'))}</td>"
            f"<td>{prettify(info.get('nationality'))}</td>"
            f"<td align='right'>{info.get('mentions', 0)}</td>"
            "</tr>"
        )
    lines.extend(["</table>", "</div>"])
    return "".join(lines)


def pie_figure(labels, counts, display_labels, hover, title, colours=None):
    colours = colours or REGION_COLOURS
    fig = go.Figure(
        data=[
            go.Pie(
                labels=display_labels,
                values=counts,
                hole=0.45,
                marker=dict(colors=[colours[i % len(colours)] for i in range(len(counts))]),
                textinfo="percent+label",
                textposition="outside",
                hovertext=hover,
                hoverinfo="text",
                customdata=labels,
            )
        ]
    )
    fig.update_layout(title=title, height=480, margin=dict(t=70, b=20, l=20, r=20))
    return fig


def subject_figure(stage_key, board_display):
    dataset = stage_dataset(stage_key)
    board_key = board_file_key(stage_key, board_display)
    subjects = subjects_for_board(stage_key, board_display)
    y_labels = [SUBJECT_LABELS.get(sb, sb.title()) for sb in subjects]
    male_c, female_c, male_s, female_s = [], [], [], []
    hover_c, hover_s = [], []
    for sb in subjects:
        cat_c = dataset[board_key]["subjects"][sb].get("concept", {"male": 0, "female": 0})
        cat_s = dataset[board_key]["subjects"][sb].get("scientist", {"male": 0, "female": 0})
        mc, fc = cat_c.get("male", 0), cat_c.get("female", 0)
        ms, fs = cat_s.get("male", 0), cat_s.get("female", 0)
        male_c.append(mc)
        female_c.append(fc)
        male_s.append(ms)
        female_s.append(fs)
        tot_c = mc + fc or 1
        tot_s = ms + fs or 1
        hover_c.append(
            f"Concept mentions<br>Men: {mc} ({mc / tot_c * 100:.1f}%)<br>Women: {fc} ({fc / tot_c * 100:.1f}%)"
        )
        hover_s.append(
            f"Scientist mentions<br>Men: {ms} ({ms / tot_s * 100:.1f}%)<br>Women: {fs} ({fs / tot_s * 100:.1f}%)"
        )
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Concept mentions", "Scientist mentions"),
        horizontal_spacing=0.12,
    )
    fig.add_trace(
        go.Bar(y=y_labels, x=male_c, name="Men", orientation="h", marker_color=MALE_COLOUR, hovertext=hover_c, hoverinfo="text"),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(y=y_labels, x=female_c, name="Women", orientation="h", marker_color=FEMALE_COLOUR, hovertext=hover_c, hoverinfo="text"),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(y=y_labels, x=male_s, name="Men ", orientation="h", marker_color=MALE_COLOUR, hovertext=hover_s, hoverinfo="text", showlegend=False),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Bar(y=y_labels, x=female_s, name="Women ", orientation="h", marker_color=FEMALE_COLOUR, hovertext=hover_s, hoverinfo="text", showlegend=False),
        row=1,
        col=2,
    )
    fig.update_layout(
        barmode="stack",
        title=f"Subject breakdown - {display_board_name(stage_key, board_display)}",
        height=max(360, 55 * len(subjects) + 120),
        margin=dict(t=80, l=140),
    )
    fig.update_xaxes(title_text="Mentions")
    return fig


def gender_figure(stage_key, board_display):
    dataset = stage_dataset(stage_key)
    board_key = board_file_key(stage_key, board_display)
    unique = dataset[board_key]["overall"]["unique"]
    male = unique.get("male", 0)
    female = unique.get("female", 0)
    total = male + female or 1
    labels = ["male", "female"]
    counts = [male, female]
    display_labels = ["Men", "Women"]
    hover = [
        f"Men<br>Count: {male}<br>Share: {male / total * 100:.1f}%",
        f"Women<br>Count: {female}<br>Share: {female / total * 100:.1f}%",
    ]
    return pie_figure(
        labels,
        counts,
        display_labels,
        hover,
        f"Unique named scientists by gender - {display_board_name(stage_key, board_display)}",
        colours=[MALE_COLOUR, FEMALE_COLOUR],
    )


def mention_type_figure(stage_key, board_display):
    dataset = stage_dataset(stage_key)
    board_key = board_file_key(stage_key, board_display)
    overall = dataset[board_key]["overall"]
    concept = overall["concept"]["male"] + overall["concept"]["female"]
    scientist = overall["scientist"]["male"] + overall["scientist"]["female"]
    total = concept + scientist or 1
    labels = ["concept", "scientist"]
    counts = [concept, scientist]
    display_labels = ["Concept", "Scientist"]
    hover = [
        f"Concept mentions<br>Count: {concept}<br>Share: {concept / total * 100:.1f}%",
        f"Scientist mentions<br>Count: {scientist}<br>Share: {scientist / total * 100:.1f}%",
    ]
    return pie_figure(
        labels,
        counts,
        display_labels,
        hover,
        f"Mention type - {display_board_name(stage_key, board_display)}",
        colours=[CONCEPT_COLOUR, SCIENTIST_COLOUR],
    )


def region_figure(stage_key, board_display=None):
    scientists = merge_scientists(stage_key, board_display)
    labels, counts, display_labels, hover, _ = demographic_counts(scientists, "region")
    scope = "all exam boards" if board_display is None else display_board_name(stage_key, board_display)
    return pie_figure(labels, counts, display_labels, hover, f"Scientists by region - {scope}")


def nationality_figure(stage_key, board_display=None):
    scientists = merge_scientists(stage_key, board_display)
    labels, counts, display_labels, hover, _ = demographic_counts(scientists, "nationality")
    scope = "all exam boards" if board_display is None else display_board_name(stage_key, board_display)
    return pie_figure(labels, counts, display_labels, hover, f"Scientists by nationality - {scope}")


def region_comparison_figure():
    scientists_g = merge_scientists("ks4")
    scientists_a = merge_scientists("ks5")
    regions_g = Counter(str(v.get("region", "unknown")).lower() for v in scientists_g.values())
    regions_a = Counter(str(v.get("region", "unknown")).lower() for v in scientists_a.values())
    all_regions = sorted(set(regions_g) | set(regions_a), key=lambda r: regions_g.get(r, 0) + regions_a.get(r, 0), reverse=True)
    total_g = sum(regions_g.values()) or 1
    total_a = sum(regions_a.values()) or 1
    pct_g = [regions_g.get(r, 0) / total_g * 100 for r in all_regions]
    pct_a = [regions_a.get(r, 0) / total_a * 100 for r in all_regions]
    y_labels = [prettify(r) for r in all_regions]
    hover_g = [
        f"KS4 - {prettify(r)}<br>Scientists: {regions_g.get(r, 0)}<br>Share: {regions_g.get(r, 0) / total_g * 100:.1f}%"
        for r in all_regions
    ]
    hover_a = [
        f"KS5 - {prettify(r)}<br>Scientists: {regions_a.get(r, 0)}<br>Share: {regions_a.get(r, 0) / total_a * 100:.1f}%"
        for r in all_regions
    ]
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            y=y_labels,
            x=pct_g,
            name="KS4 (GCSE / NQ5)",
            orientation="h",
            marker_color=pf.PURPLE,
            hovertext=hover_g,
            hoverinfo="text",
            customdata=all_regions,
        )
    )
    fig.add_trace(
        go.Bar(
            y=y_labels,
            x=pct_a,
            name="KS5 (A-Level / Highers)",
            orientation="h",
            marker_color=pf.PURPLE_LIGHT,
            hovertext=hover_a,
            hoverinfo="text",
            customdata=all_regions,
        )
    )
    fig.update_layout(
        barmode="group",
        title="Regional representation - KS4 vs KS5 (% of unique scientists)",
        xaxis_title="Percentage (%)",
        height=max(420, 42 * len(all_regions) + 140),
        margin=dict(l=160, t=70),
    )
    return fig


def build_figure(stage_key, board_display, view_name):
    if view_name == "Subject breakdown (by gender)":
        return subject_figure(stage_key, board_display)
    if view_name == "Unique scientists by gender":
        return gender_figure(stage_key, board_display)
    if view_name == "Concept vs scientist mentions":
        return mention_type_figure(stage_key, board_display)
    if view_name == "Scientists by region":
        return region_figure(stage_key, board_display)
    if view_name == "Scientists by nationality":
        return nationality_figure(stage_key, board_display)
    if view_name == "KS4 vs KS5 region comparison":
        return region_comparison_figure()
    raise ValueError(view_name)


def summary_html(stage_key, board_display, view_name):
    scientists = merge_scientists(stage_key, board_display)
    total = len(scientists)
    women = sum(1 for s in scientists.values() if str(s.get("gender", "")).lower() == "female")
    men = sum(1 for s in scientists.values() if str(s.get("gender", "")).lower() == "male")
    board_label = display_board_name(stage_key, board_display)
    pct_women = women / total * 100 if total else 0
    return (
        f"<div style='padding:10px 14px; background:#f0f6fc; border-radius:8px; margin-bottom:10px;'>"
        f"<strong>{board_label}</strong> | {total} unique scientists | "
        f"{women} women ({pct_women:.1f}%) | {men} men | View: {view_name}"
        f"</div>"
    )


In [ ]:
stage_dropdown = widgets.Dropdown(
    options=list(KEY_STAGE_OPTIONS.keys()),
    value=list(KEY_STAGE_OPTIONS.keys())[0],
    description="Key stage:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

board_dropdown = widgets.Dropdown(
    description="Exam board:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

view_dropdown = widgets.Dropdown(
    options=VIEW_OPTIONS,
    value=VIEW_OPTIONS[0],
    description="Chart view:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

demo_dimension_dropdown = widgets.Dropdown(
    options=list(DEMOGRAPHIC_OPTIONS.keys()),
    value="Gender",
    description="Explore by:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

demo_value_dropdown = widgets.Dropdown(
    description="Group:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

summary_out = widgets.Output()
names_out = widgets.Output()
figure_widget = go.FigureWidget()


def refresh_board_options(*_):
    stage_key = KEY_STAGE_OPTIONS[stage_dropdown.value]
    boards = stage_cfg(stage_key)["boards_display"]
    board_dropdown.options = boards
    if board_dropdown.value not in boards:
        board_dropdown.value = boards[0]


def refresh_demo_values(*_):
    stage_key = KEY_STAGE_OPTIONS[stage_dropdown.value]
    scientists = merge_scientists(stage_key, board_dropdown.value)
    dimension = DEMOGRAPHIC_OPTIONS[demo_dimension_dropdown.value]
    values = demographic_values(scientists, dimension)
    demo_value_dropdown.options = [(prettify(v), v) for v in values]
    if values:
        demo_value_dropdown.value = values[0]


def show_names_for_selection(dimension=None, raw_value=None, extra_title=""):
    stage_key = KEY_STAGE_OPTIONS[stage_dropdown.value]
    scientists = merge_scientists(stage_key, board_dropdown.value)
    kwargs = {}
    if dimension and raw_value is not None:
        kwargs[dimension] = raw_value
    rows = filter_scientists(scientists, **kwargs)
    if dimension and raw_value is not None:
        title = f"{prettify(raw_value)} ({prettify(dimension)}){extra_title}"
    else:
        title = f"All scientists - {display_board_name(stage_key, board_dropdown.value)}{extra_title}"
    with names_out:
        names_out.clear_output(wait=True)
        display(HTML(names_html(rows, title)))


def attach_click_handlers(*_):
    for trace in figure_widget.data:
        trace.on_click(on_chart_click)


def refresh_chart(*_):
    stage_key = KEY_STAGE_OPTIONS[stage_dropdown.value]
    view_name = view_dropdown.value
    comparison = view_name == "KS4 vs KS5 region comparison"
    board_dropdown.disabled = comparison
    with summary_out:
        summary_out.clear_output(wait=True)
        if comparison:
            display(HTML("<div style='padding:10px 14px; background:#f0f6fc; border-radius:8px; margin-bottom:10px;'>Comparing regional representation across both key stages.</div>"))
        else:
            display(HTML(summary_html(stage_key, board_dropdown.value, view_name)))
    fig = build_figure(stage_key, board_dropdown.value, view_name)
    figure_widget.data = []
    figure_widget.layout = fig.layout
    for trace in fig.data:
        figure_widget.add_trace(trace)
    attach_click_handlers()
    refresh_demo_values()
    show_names_for_selection()


def on_chart_click(trace, points, selector):
    if not points.point_inds:
        return
    idx = points.point_inds[0]
    view_name = view_dropdown.value
    if view_name == "Unique scientists by gender":
        raw = trace.customdata[idx] if trace.customdata is not None else ["male", "female"][idx]
        show_names_for_selection("gender", raw, " (clicked on chart)")
        demo_dimension_dropdown.value = "Gender"
        demo_value_dropdown.value = raw
    elif view_name in ("Scientists by region", "Scientists by nationality"):
        raw = trace.customdata[idx]
        dimension = "region" if view_name == "Scientists by region" else "nationality"
        show_names_for_selection(dimension, raw, " (clicked on chart)")
        demo_dimension_dropdown.value = "Region" if dimension == "region" else "Nationality"
        demo_value_dropdown.value = raw
    elif view_name == "KS4 vs KS5 region comparison":
        raw = trace.customdata[idx]
        label = "KS4" if trace.name.startswith("KS4") else "KS5"
        stage_for_names = "ks4" if label == "KS4" else "ks5"
        scientists = merge_scientists(stage_for_names)
        rows = filter_scientists(scientists, region=raw)
        title = f"{prettify(raw)} (Region) - {label} (clicked on chart)"
        with names_out:
            names_out.clear_output(wait=True)
            display(HTML(names_html(rows, title)))
    elif view_name == "Concept vs scientist mentions":
        raw = trace.customdata[idx] if trace.customdata is not None else ["concept", "scientist"][idx]
        with names_out:
            names_out.clear_output(wait=True)
            display(HTML(
                f"<p><em>Mention-type charts count <strong>all mentions</strong>, not unique people. "
                f"Selected: <strong>{prettify(raw)}</strong>. Use the demographic explorer below to list individuals.</em></p>"
            ))


def on_demo_change(*_):
    dimension = DEMOGRAPHIC_OPTIONS[demo_dimension_dropdown.value]
    raw_value = demo_value_dropdown.value
    show_names_for_selection(dimension, raw_value)


stage_dropdown.observe(refresh_board_options, names="value")
stage_dropdown.observe(refresh_chart, names="value")
board_dropdown.observe(refresh_chart, names="value")
board_dropdown.observe(refresh_demo_values, names="value")
view_dropdown.observe(refresh_chart, names="value")
demo_dimension_dropdown.observe(refresh_demo_values, names="value")
demo_dimension_dropdown.observe(on_demo_change, names="value")
demo_value_dropdown.observe(on_demo_change, names="value")

refresh_board_options()
refresh_chart()

controls = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:8px'>Filters</h3>"),
    widgets.HBox([stage_dropdown, board_dropdown, view_dropdown], layout=widgets.Layout(flex_wrap="wrap")),
    widgets.HTML("<h3 style='margin:18px 0 8px'>Demographic explorer</h3><p style='margin-top:0;color:#555'>Pick a group to list every named scientist, or click a chart segment above.</p>"),
    widgets.HBox([demo_dimension_dropdown, demo_value_dropdown], layout=widgets.Layout(flex_wrap="wrap")),
])

dashboard = widgets.VBox([
    controls,
    summary_out,
    figure_widget,
    widgets.HTML("<h3 style='margin:18px 0 8px'>Scientists in selected group</h3>"),
    names_out,
])

display(dashboard)
